# 表达式转换

栈的一个重要应用是对表达式进行转换。例如，将中缀表达式转换为后缀表达式，使之便于计算机处理。在前缀表达式和后缀表达式中，运算顺序已经由符号的位置隐含确定，不再依赖复杂的括号。

常见的三种表达式写法如下：

| 类型 | 运算符位置 | 例子 |
| --- | --- | --- |
| 前缀表达式 | 运算符在操作数前面 | `* + A B C` |
| 中缀表达式 | 运算符在两个操作数中间 | `(A + B) * C` |
| 后缀表达式 | 运算符在操作数后面 | `A B + C *` |

中缀表达式最符合人类习惯，但需要处理优先级和括号；前缀、后缀表达式更适合计算机用栈直接求值。

## 表达式树

任意一个表达式都可以视作一棵表达式树的某种遍历结果。例如，

```text
(A + B) * C
```

对应的表达式树是

```text
      *
     / \
    +   C
   / \
  A   B
```

三种遍历方式分别对应三种表达式：

前序遍历：根 $\to$ 左 $\to$ 右

```text
* + A B C
```

中序遍历：左 $\to$ 根 $\to$ 右

```text
(A + B) * C
```

后序遍历：左 $\to$ 右 $\to$ 根

```text
A B + C *
```

因此，表达式转换可以被理解为两步：先把表达式还原成树，再对树做不同遍历。

## 记号、优先级与基本工具

下面的代码约定：

- 支持二元运算符 `+ - * / ^`；
- `^` 表示乘方，并且是右结合的，即 `2 ^ 3 ^ 2` 等价于 `2 ^ (3 ^ 2)`；
- 中缀表达式可以写成普通字符串，如 `"(A + B) * C"`；
- 前缀、后缀表达式默认用空格分隔 token，如 `"* + A B C"`；
- 为了让主线更清晰，这里主要处理二元运算。负数数字字面量如 `-3` 可以识别，但 `-x` 这类一元负号不单独处理。

In [30]:
OPERATORS = {
    "+": (1, "L"),
    "-": (1, "L"),
    "*": (2, "L"),
    "/": (2, "L"),
    "^": (3, "R"),
}


def is_operator(token):
    return token in OPERATORS


def is_number(token):
    try:
        float(token)
        return True
    except ValueError:
        return False


def is_operand(token):
    return token not in OPERATORS and token not in {"(", ")"}


def tokenize(expr):
    """把中缀表达式字符串切分成 token 列表。"""
    tokens = []
    i = 0
    prev = None

    while i < len(expr):
        ch = expr[i]

        if ch.isspace():
            i += 1
            continue

        is_negative_number = (
            ch == "-"
            and (prev is None or prev in OPERATORS or prev == "(")
            and i + 1 < len(expr)
            and (expr[i + 1].isdigit() or expr[i + 1] == ".")
        )

        if ch.isdigit() or ch == "." or is_negative_number:
            start = i
            if is_negative_number:
                i += 1

            dot_count = 0
            while i < len(expr) and (expr[i].isdigit() or expr[i] == "."):
                if expr[i] == ".":
                    dot_count += 1
                if dot_count > 1:
                    raise ValueError(f"非法数字: {expr[start:i + 1]}")
                i += 1

            tokens.append(expr[start:i])
            prev = "operand"
            continue

        if ch.isalpha() or ch == "_":
            start = i
            while i < len(expr) and (expr[i].isalnum() or expr[i] == "_"):
                i += 1
            tokens.append(expr[start:i])
            prev = "operand"
            continue

        if ch in "+-*/^()":
            tokens.append(ch)
            prev = ch
            i += 1
            continue

        raise ValueError(f"无法识别的字符: {ch}")

    return tokens


tokenize("(A + B) * C")

['(', 'A', '+', 'B', ')', '*', 'C']

## 中缀表达式转后缀表达式

中缀表达式转后缀表达式常用 **调度场算法**（shunting-yard algorithm）。核心思想是：

- 操作数直接输出；
- 左括号入栈；
- 右括号触发弹栈，直到遇到左括号；
- 运算符入栈前，先把栈顶优先级更高的运算符弹出；如果二者优先级相同，还要看当前运算符是否左结合，相同优先级左结合的运算符从左到右计算，因此应当弹出栈顶运算符；
- 扫描结束后，把栈中剩余运算符依次弹出。

这相当于把中缀表达式中隐含的运算顺序显式地移动到运算符的位置上。

In [31]:
def infix_to_postfix(expr):
    tokens = tokenize(expr) if isinstance(expr, str) else list(expr)
    output = []
    stack = []

    for token in tokens:
        if is_operand(token):
            output.append(token)
        elif token == "(":
            stack.append(token)
        elif token == ")":
            while stack and stack[-1] != "(":
                output.append(stack.pop())
            if not stack:
                raise ValueError("括号不匹配")
            stack.pop()
        elif is_operator(token):
            while (
                stack
                and stack[-1] in OPERATORS
                and (
                    (OPERATORS[token][0] < OPERATORS[stack[-1]][0])
                    or (
                        OPERATORS[token][0] == OPERATORS[stack[-1]][0]
                        and OPERATORS[token][1] == "L"
                    )
                )
            ):
                output.append(stack.pop())
            stack.append(token)
        else:
            raise ValueError(f"无法识别的 token: {token}")

    while stack:
        if stack[-1] in "()":
            raise ValueError("括号不匹配")
        output.append(stack.pop())

    return output
        

postfix = infix_to_postfix("(A + B) * C")
postfix, " ".join(postfix)

(['A', 'B', '+', 'C', '*'], 'A B + C *')

## 中缀表达式转前缀表达式
中缀表达式转前缀表达式与中缀表达式转后缀表达式的思路很相近，但是需要注意的是，由于操作符在操作数之前，我们应当从右往左扫描 token 序列：
- 操作数直接添加到输出列表中；
- 右括号入栈；
- 左括号触发弹栈，直至遇到下一个右括号；
- 运算符入栈前，如果其优先级低于栈顶运算符的优先级，则应当弹栈；
- 如果其优先级与栈顶运算符优先级相同，且该运算符是右结合的，则应当先弹栈；
- 扫描结束后，依次弹出栈中剩余操作符。

In [32]:
def infix_to_prefix(expr):
    tokens = tokenize(expr) if isinstance(expr, str) else list(expr)
    output = []
    stack = []

    for token in tokens[::-1]:
        if is_operand(token):
            output.append(token)
        elif token == ")":
            stack.append(token)
        elif token == "(":
            while stack and stack[-1] != ")":
                output.append(stack.pop())
            if not stack:
                raise ValueError("括号不匹配")
            stack.pop()
        elif is_operator(token):
            while (
                stack
                and stack[-1] in OPERATORS
                and (
                    (OPERATORS[token][0] < OPERATORS[stack[-1]][0])
                    or (
                        OPERATORS[token][0] == OPERATORS[stack[-1]][0]
                        and OPERATORS[token][1] == "R"
                    )
                )
            ):
                output.append(stack.pop())
            stack.append(token)
        else:
            raise ValueError(f"无法识别的 token: {token}")

    while stack:
        if stack[-1] in "()":
            raise ValueError("括号不匹配")
        output.append(stack.pop())

    return output[::-1]
        

prefix = infix_to_prefix("(A + B) * C")
prefix, " ".join(prefix)

(['*', '+', 'A', 'B', 'C'], '* + A B C')

## 从后缀或前缀表达式还原中缀表达式

前缀、后缀表达式不需要括号，是因为每个运算符都能找到自己的两个操作数。利用栈可以直接把它们还原为带括号的中缀表达式。

后缀表达式从左到右扫描：

- 遇到操作数，压栈；
- 遇到运算符，弹出两个表达式，先弹出的是右操作数，后弹出的是左操作数；
- 合成新的中缀表达式后再压栈。

前缀表达式可以反过来从右到左扫描：

- 遇到操作数，压栈；
- 遇到运算符，弹出两个表达式，先弹出的是左操作数，后弹出的是右操作数；
- 合成新的中缀表达式后再压栈。

In [33]:
def postfix_to_infix(expr):
    tokens = expr.split() if isinstance(expr, str) else list(expr)
    stack = []

    for token in tokens:
        if is_operand(token):
            stack.append(token)
        elif is_operator(token):
            if len(stack) < 2:
                raise ValueError("表达式不完整")
            right = stack.pop()
            left = stack.pop()
            stack.append(f"({left} {token} {right})")

    if len(stack) != 1:
        raise ValueError("表达式不合法")
    return stack[0]


def prefix_to_infix(expr):
    tokens = expr.split() if isinstance(expr, str) else list(expr)
    stack = []

    for token in reversed(tokens):
        if is_operand(token):
            stack.append(token)
        elif is_operator(token):
            if len(stack) < 2:
                raise ValueError("表达式不完整")
            left = stack.pop()
            right = stack.pop()
            stack.append(f"({left} {token} {right})")

    if len(stack) != 1:
        raise ValueError("表达式不合法")
    return stack[0]


postfix_to_infix("A B + C *"), prefix_to_infix("* + A B C")

('((A + B) * C)', '((A + B) * C)')

## 用表达式树统一转换

如果直接写“前缀转后缀”“后缀转前缀”“中缀转前缀”等函数，很容易出现大量相似代码。表达式树可以把转换过程简化为：

```text
任意一种表达式 -> 表达式树 -> 任意一种遍历结果
```

于是只需要实现三类“建树”和三类“遍历”，就能覆盖所有转换。

In [34]:
from dataclasses import dataclass


@dataclass
class ExprNode:
    value: str
    left: object = None
    right: object = None

    def is_leaf(self):
        return self.left is None and self.right is None


def tree_from_postfix(expr):
    tokens = expr.split() if isinstance(expr, str) else list(expr)
    stack = []

    for token in tokens:
        if is_operand(token):
            stack.append(ExprNode(token))
        elif is_operator(token):
            if len(stack) < 2:
                raise ValueError("表达式不完整")
            right = stack.pop()
            left = stack.pop()
            stack.append(ExprNode(token, left, right))

    if len(stack) != 1:
        raise ValueError("表达式不合法")
    return stack[0]


def tree_from_prefix(expr):
    tokens = expr.split() if isinstance(expr, str) else list(expr)
    stack = []

    for token in reversed(tokens):
        if is_operand(token):
            stack.append(ExprNode(token))
        elif is_operator(token):
            if len(stack) < 2:
                raise ValueError("表达式不完整")
            left = stack.pop()
            right = stack.pop()
            stack.append(ExprNode(token, left, right))

    if len(stack) != 1:
        raise ValueError("表达式不合法")
    return stack[0]


def tree_from_infix(expr):
    return tree_from_postfix(infix_to_postfix(expr))

In [35]:
def tree_to_prefix(node):
    if node.is_leaf():
        return [node.value]
    return [node.value] + tree_to_prefix(node.left) + tree_to_prefix(node.right)


def tree_to_postfix(node):
    if node.is_leaf():
        return [node.value]
    return tree_to_postfix(node.left) + tree_to_postfix(node.right) + [node.value]


def tree_to_infix(node):
    if node.is_leaf():
        return node.value
    return f"({tree_to_infix(node.left)} {node.value} {tree_to_infix(node.right)})"


def build_tree(expr, notation):
    if notation == "infix":
        return tree_from_infix(expr)
    if notation == "prefix":
        return tree_from_prefix(expr)
    if notation == "postfix":
        return tree_from_postfix(expr)
    raise ValueError("notation 必须是 infix、prefix 或 postfix")


def convert(expr, source, target):
    tree = build_tree(expr, source)
    if target == "infix":
        return tree_to_infix(tree)
    if target == "prefix":
        return " ".join(tree_to_prefix(tree))
    if target == "postfix":
        return " ".join(tree_to_postfix(tree))
    raise ValueError("target 必须是 infix、prefix 或 postfix")

In [36]:
expr = "(A + B) * C"

print("infix  -> prefix :", convert(expr, "infix", "prefix"))
print("infix  -> postfix:", convert(expr, "infix", "postfix"))
print("prefix -> infix  :", convert("* + A B C", "prefix", "infix"))
print("postfix-> prefix :", convert("A B + C *", "postfix", "prefix"))

infix  -> prefix : * + A B C
infix  -> postfix: A B + C *
prefix -> infix  : ((A + B) * C)
postfix-> prefix : * + A B C


## 前缀、后缀表达式求值

后缀表达式求值和后缀还原中缀非常相似，只是栈中保存的不再是字符串，而是数值：

- 遇到数字或变量，取出它的值并压栈；
- 遇到运算符，弹出两个数，计算结果后再压栈；
- 扫描结束后，栈中唯一元素就是表达式的值。

前缀表达式同理，只是从右往左扫描。中缀表达式可以先转成后缀再求值，也可以先建表达式树再递归求值。

In [37]:
def apply_operator(op, left, right):
    if op == "+":
        return left + right
    if op == "-":
        return left - right
    if op == "*":
        return left * right
    if op == "/":
        return left / right
    if op == "^":
        return left ** right
    raise ValueError(f"未知运算符: {op}")


def value_of(token, env=None):
    if is_number(token):
        value = float(token)
        return int(value) if value.is_integer() else value

    env = env or {}
    if token in env:
        return env[token]

    raise ValueError(f"变量 {token} 没有给定取值")

In [38]:
def eval_postfix(expr, env=None):
    tokens = expr.split() if isinstance(expr, str) else list(expr)
    stack = []

    for token in tokens:
        if is_operand(token):
            stack.append(value_of(token, env))
        elif is_operator(token):
            if len(stack) < 2:
                raise ValueError("表达式不完整")
            right = stack.pop()
            left = stack.pop()
            stack.append(apply_operator(token, left, right))

    if len(stack) != 1:
        raise ValueError("表达式不合法")
    return stack[0]


def eval_prefix(expr, env=None):
    tokens = expr.split() if isinstance(expr, str) else list(expr)
    stack = []

    for token in reversed(tokens):
        if is_operand(token):
            stack.append(value_of(token, env))
        elif is_operator(token):
            if len(stack) < 2:
                raise ValueError("表达式不完整")
            left = stack.pop()
            right = stack.pop()
            stack.append(apply_operator(token, left, right))

    if len(stack) != 1:
        raise ValueError("表达式不合法")
    return stack[0]


eval_postfix("3 4 + 5 *"), eval_prefix("* + 3 4 5")

(35, 35)

## 用表达式树求值

表达式树求值更接近递归定义：

- 如果当前结点是叶子结点，它就是操作数，直接取值；
- 如果当前结点是运算符，就先求左子树的值，再求右子树的值，最后把二者合并。

这说明表达式树不仅能统一转换，也能统一求值。

In [39]:
def eval_tree(node, env=None):
    if node.is_leaf():
        return value_of(node.value, env)

    left = eval_tree(node.left, env)
    right = eval_tree(node.right, env)
    return apply_operator(node.value, left, right)


def evaluate(expr, notation="infix", env=None):
    tree = build_tree(expr, notation)
    return eval_tree(tree, env)


print(evaluate("(3 + 4) * 5", "infix"))
print(evaluate("* + A B C", "prefix", {"A": 3, "B": 4, "C": 5}))
print(evaluate("A B + C *", "postfix", {"A": 3, "B": 4, "C": 5}))

35
35
35


## 小结

- 中缀表达式适合人读，但程序处理时要考虑括号、优先级和结合性；
- 后缀表达式适合从左到右用栈求值；
- 前缀表达式适合从右到左用栈求值；
- 表达式树把三种写法统一起来：前缀是前序遍历，中缀是中序遍历，后缀是后序遍历；
- 用树做中间表示后，任意表达式转换都可以写成“建树 + 遍历”，求值也可以写成对树的递归。

In [40]:
# 综合例子
expr = "3 + 4 * 2 / (1 - 5) ^ 2 ^ 3"

prefix = convert(expr, "infix", "prefix")
postfix = convert(expr, "infix", "postfix")

print("中缀:", expr)
print("前缀:", prefix)
print("后缀:", postfix)
print("值  :", evaluate(expr, "infix"))

中缀: 3 + 4 * 2 / (1 - 5) ^ 2 ^ 3
前缀: + 3 / * 4 2 ^ - 1 5 ^ 2 3
后缀: 3 4 2 * 1 5 - 2 3 ^ ^ / +
值  : 3.0001220703125
